# Experiment: FLUX.2 VAE round-trip audit over all ARC training outputs

This Colab experiment measures the clean frozen-VAE reconstruction ceiling for
every labelled output in the ARC-AGI-2 `training` directory using the production
`greedy_rgb_maximin_v1` palette and the fixed top-left-aligned 30×30 canvas.

Scope and success criteria:

- audit all 1,000 task files and all 4,308 outputs exactly once in the production path;
- report exact-grid, inferred-shape, true-grid-cell, full-canvas, and PAD metrics;
- retain every failure with task/split/index identity and cell-level differences;
- rerun production failures on their minimum square canvas to diagnose padding sensitivity;
- save a reproducibility report, flat tables, detailed JSONL, and bounded gallery pages to Drive.

This is a representation diagnostic only. It does not train a model, alter the
palette, add augmentations, or evaluate ARC input grids.

## Colab prerequisites

Select an A100 GPU runtime and provide these files in Google Drive:

```text
MyDrive/diffusion-arc/
├── models/flux2-klein-base-4b/vae/
│   ├── config.json
│   └── diffusion_pytorch_model.safetensors
└── data/raw/arc-agi-2/training/
    └── <1,000 task JSON files>
```

A `GITHUB_TOKEN` Colab secret is optional for fetching the pinned public source.

In [ ]:
!nvidia-smi

## 1. Fixed runtime and experiment identity

The source revision, VAE hash, expected dataset counts, palette, geometry, and
physical batch sizes are fixed before examining any reconstruction result.

In [ ]:
from __future__ import annotations

import base64
import csv
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import torch
from google.colab import drive, userdata
from IPython.display import display
from tqdm.auto import tqdm

drive.mount('/content/drive')

SEED = 20260826
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = 'ae1bc579844a67ea214f9389447fca7c6f6efbc1'
EXPECTED_VAE_SHA256 = 'ca70d2202afe6415bdbcb8793ba8cd99fd159cfe6192381504d6c4d3036e0f04'
EXPECTED_TARGET_PALETTE_ID = 'greedy_rgb_maximin_v1'
EXPECTED_TASK_COUNT = 1_000
EXPECTED_TRAIN_OUTPUT_COUNT = 3_232
EXPECTED_TEST_OUTPUT_COUNT = 1_076
EXPECTED_OUTPUT_COUNT = 4_308
ENCODE_BATCH_SIZE = 16
DECODE_BATCH_SIZE = 8
MAX_GALLERY_FAILURES = 64
GALLERY_ROWS_PER_PAGE = 12

DRIVE_ROOT = Path('/content/drive/MyDrive/diffusion-arc')
VAE_SOURCE_PATH = DRIVE_ROOT / 'models' / 'flux2-klein-base-4b' / 'vae'
TRAINING_DATA_DIR = DRIVE_ROOT / 'data' / 'raw' / 'arc-agi-2' / 'training'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / 'vae-training-output-roundtrip-audit' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-vae-training-audit-source-{RUN_ID}'
LOCAL_VAE_PATH = Path('/tmp') / f'flux2-klein-base-4b-vae-{RUN_ID}'

if not torch.cuda.is_available():
    raise RuntimeError('This audit requires a CUDA GPU runtime')
DEVICE = torch.device('cuda')
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
gpu = torch.cuda.get_device_properties(DEVICE)

print(json.dumps({
    'run_id': RUN_ID,
    'gpu': gpu.name,
    'gpu_memory_GiB': round(gpu.total_memory / 2**30, 1),
    'source_revision': SOURCE_REVISION,
    'training_data_dir': str(TRAINING_DATA_DIR),
    'output_dir': str(OUTPUT_DIR),
    'encode_batch_size': ENCODE_BATCH_SIZE,
    'decode_batch_size': DECODE_BATCH_SIZE,
}, indent=2))

## 2. Fetch the exact source and validate its Python path

Colab currently uses Python 3.13, outside the repository's packaged-support
range. The notebook therefore imports the pinned checkout directly through
`PYTHONPATH`; it does not weaken or bypass the package metadata.

In [ ]:
def run_command(
    label: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-60:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


SOURCE_DIR.mkdir(parents=True, exist_ok=False)
run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
run_command('add origin', ['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=SOURCE_DIR)
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = os.environ.get('GITHUB_TOKEN')
github_token = (github_token or '').strip()
git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
if github_token:
    credential = base64.b64encode(
        f'x-access-token:{github_token}'.encode('utf-8')
    ).decode('ascii')
    git_environment.update({
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
    })
try:
    run_command(
        'fetch source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
        cwd=SOURCE_DIR,
        env=git_environment,
    )
    run_command('checkout source', ['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=SOURCE_DIR)
finally:
    git_environment.clear()
    github_token = None
    if 'credential' in locals():
        del credential

RESOLVED_SOURCE_COMMIT = run_command('resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR)
assert RESOLVED_SOURCE_COMMIT == SOURCE_REVISION

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}
def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


requirements = [
    f'{name}=={version}'
    for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command(
        'install pinned dependencies',
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', '--no-cache-dir', *requirements],
    )

SOURCE_PATH = str((SOURCE_DIR / 'src').resolve())
if SOURCE_PATH not in sys.path:
    sys.path.insert(0, SOURCE_PATH)
importlib.invalidate_caches()
print(f'Using pinned Diffusion-ARC source directly: {SOURCE_PATH}')

test_environment = os.environ.copy()
test_environment.update({'PYTHONDONTWRITEBYTECODE': '1', 'PYTHONPATH': SOURCE_PATH})
tests_started = time.perf_counter()
tests = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR,
    env=test_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
print('\n'.join(tests.stdout.splitlines()[-35:]))
if tests.returncode != 0:
    raise RuntimeError(f'Repository unit tests returned {tests.returncode}')
print({
    'resolved_source_commit': RESOLVED_SOURCE_COMMIT,
    'unit_tests_seconds': round(time.perf_counter() - tests_started, 1),
})

## 3. Load the frozen VAE and inventory all labelled outputs

`load_training_task` validates the ARC files through the production loader.
Split identity and example indices are retained separately so failures remain
traceable to `train[i].output` or `test[i].output`.

In [ ]:
from diffusers import AutoencoderKLFlux2
from diffusion_arc.arc_data import (
    CANVAS_SIZE,
    CELL_SIZE,
    PAD_CLASS,
    TARGET_PALETTE_ID,
    TARGET_PALETTE_RGB,
    classify_arc_cells,
    decode_arc_grids,
    load_training_task,
    render_grid_canvas,
)
from diffusion_arc.model import FluxSpatialShape, FluxVaeDecoder, FluxVaeEncoder

if TARGET_PALETTE_ID != EXPECTED_TARGET_PALETTE_ID:
    raise RuntimeError(
        f'Expected palette {EXPECTED_TARGET_PALETTE_ID}, received {TARGET_PALETTE_ID}'
    )

required_vae_files = ('config.json', 'diffusion_pytorch_model.safetensors')
missing_vae_files = [
    name for name in required_vae_files if not (VAE_SOURCE_PATH / name).is_file()
]
if missing_vae_files:
    raise FileNotFoundError(f'Missing VAE files in Drive: {missing_vae_files}')
vae_sha256 = sha256_file(VAE_SOURCE_PATH / 'diffusion_pytorch_model.safetensors')
if vae_sha256 != EXPECTED_VAE_SHA256:
    raise RuntimeError(
        f'Unexpected VAE hash: expected {EXPECTED_VAE_SHA256}, received {vae_sha256}'
    )
shutil.copytree(VAE_SOURCE_PATH, LOCAL_VAE_PATH)

vae = AutoencoderKLFlux2.from_pretrained(
    LOCAL_VAE_PATH,
    local_files_only=True,
    torch_dtype=torch.float32,
).to(DEVICE)
vae.eval().requires_grad_(False)
vae_encoder = FluxVaeEncoder(vae)
vae_decoder = FluxVaeDecoder(vae)
assert not vae.training


@dataclass(frozen=True)
class OutputRecord:
    record_id: str
    task_id: str
    split: str
    example_index: int
    grid: tuple[tuple[int, ...], ...]

    @property
    def height(self) -> int:
        return len(self.grid)

    @property
    def width(self) -> int:
        return len(self.grid[0])


task_paths = sorted(TRAINING_DATA_DIR.glob('*.json'))
if len(task_paths) != EXPECTED_TASK_COUNT:
    raise RuntimeError(
        f'Expected {EXPECTED_TASK_COUNT} training task files, found {len(task_paths)}'
    )

dataset_digest = hashlib.sha256()
task_manifest = []
records: list[OutputRecord] = []
split_counts: Counter[str] = Counter()
shape_counts: Counter[tuple[int, int]] = Counter()
color_counts: Counter[int] = Counter()

for task_path in tqdm(task_paths, desc='Validate ARC task files', unit='task'):
    raw_bytes = task_path.read_bytes()
    file_sha256 = hashlib.sha256(raw_bytes).hexdigest()
    dataset_digest.update(task_path.name.encode('utf-8'))
    dataset_digest.update(b'\0')
    dataset_digest.update(bytes.fromhex(file_sha256))
    payload = json.loads(raw_bytes)
    task = load_training_task(task_path)
    train_count = len(payload['train'])
    test_count = len(payload['test'])
    if len(task.pairs) != train_count + test_count:
        raise RuntimeError(f'Loaded pair count mismatch for {task.task_id}')
    task_manifest.append({
        'task_id': task.task_id,
        'sha256': file_sha256,
        'train_outputs': train_count,
        'test_outputs': test_count,
    })
    for pair_index, pair in enumerate(task.pairs):
        split = 'train' if pair_index < train_count else 'test'
        example_index = pair_index if split == 'train' else pair_index - train_count
        record = OutputRecord(
            record_id=f'{task.task_id}:{split}:{example_index}',
            task_id=task.task_id,
            split=split,
            example_index=example_index,
            grid=pair.output_grid,
        )
        records.append(record)
        split_counts[split] += 1
        shape_counts[(record.height, record.width)] += 1
        color_counts.update(value for row in record.grid for value in row)

if split_counts != Counter({
    'train': EXPECTED_TRAIN_OUTPUT_COUNT,
    'test': EXPECTED_TEST_OUTPUT_COUNT,
}):
    raise RuntimeError(f'Unexpected output split counts: {dict(split_counts)}')
if len(records) != EXPECTED_OUTPUT_COUNT:
    raise RuntimeError(f'Expected {EXPECTED_OUTPUT_COUNT} outputs, found {len(records)}')

DATASET_SHA256 = dataset_digest.hexdigest()
dataset_summary = {
    'task_count': len(task_paths),
    'output_count': len(records),
    'split_counts': dict(sorted(split_counts.items())),
    'dataset_sha256': DATASET_SHA256,
    'height_range': [min(record.height for record in records), max(record.height for record in records)],
    'width_range': [min(record.width for record in records), max(record.width for record in records)],
    'area_range': [
        min(record.height * record.width for record in records),
        max(record.height * record.width for record in records),
    ],
    'unique_shapes': len(shape_counts),
    'color_counts': {str(key): value for key, value in sorted(color_counts.items())},
}
print(json.dumps({
    'vae_class': type(vae).__name__,
    'vae_parameters': sum(parameter.numel() for parameter in vae.parameters()),
    'vae_sha256': vae_sha256,
    'target_palette': TARGET_PALETTE_ID,
    'dataset': dataset_summary,
}, indent=2))

## 4. Production-faithful round-trip helpers

Exact-grid is strict: the PAD boundary must infer the right shape and every
target cell must match. Full-canvas and PAD metrics are retained only as
diagnostics because white padding dominates small grids.

In [ ]:
PRODUCTION_SPATIAL_SHAPE = FluxSpatialShape(height=CANVAS_SIZE, width=CANVAS_SIZE)
PALETTE_FLOAT = TARGET_PALETTE_RGB.to(dtype=torch.float32)


def normalized_images_to_uint8(images: torch.Tensor) -> torch.Tensor:
    return images.detach().clamp(-1, 1).add(1).mul(127.5).round().to(
        device='cpu', dtype=torch.uint8
    )


def expected_class_canvas(
    record: OutputRecord,
    *,
    canvas_size: int,
) -> torch.Tensor:
    canvas = torch.full((canvas_size, canvas_size), PAD_CLASS, dtype=torch.int64)
    canvas[:record.height, :record.width] = torch.tensor(record.grid, dtype=torch.int64)
    return canvas


def cell_means_uint8(rgb_image: torch.Tensor, *, canvas_size: int) -> torch.Tensor:
    return rgb_image.to(dtype=torch.float32).reshape(
        3, canvas_size, CELL_SIZE, canvas_size, CELL_SIZE
    ).mean(dim=(2, 4)).permute(1, 2, 0)


def analyze_reconstruction(
    record: OutputRecord,
    rgb_image: torch.Tensor,
    class_canvas: torch.Tensor,
    prediction: Any,
    *,
    canvas_size: int,
) -> dict[str, Any]:
    expected = expected_class_canvas(record, canvas_size=canvas_size)
    predicted = class_canvas.to(device='cpu', dtype=torch.int64)
    matches = predicted == expected
    true_grid_matches = matches[:record.height, :record.width]
    expected_pad = expected == PAD_CLASS
    predicted_pad = predicted == PAD_CLASS
    pad_true_positive = int((expected_pad & predicted_pad).sum().item())
    pad_false_positive = int((~expected_pad & predicted_pad).sum().item())
    pad_false_negative = int((expected_pad & ~predicted_pad).sum().item())
    predicted_shape = (
        None
        if prediction.height is None or prediction.width is None
        else [int(prediction.height), int(prediction.width)]
    )
    shape_exact = predicted_shape == [record.height, record.width]
    exact_grid = bool(shape_exact and true_grid_matches.all().item())

    means = cell_means_uint8(rgb_image, canvas_size=canvas_size)
    all_distances = (means.unsqueeze(2) - PALETTE_FLOAT.reshape(1, 1, -1, 3)).square().sum(dim=-1)
    mismatch_positions = torch.nonzero(~matches, as_tuple=False)
    mismatches = []
    for row_index, column_index in mismatch_positions.tolist():
        distances = all_distances[row_index, column_index]
        sorted_distances, sorted_indices = torch.sort(distances)
        expected_class = int(expected[row_index, column_index].item())
        predicted_class = int(predicted[row_index, column_index].item())
        mismatches.append({
            'row': row_index,
            'column': column_index,
            'region': 'grid' if row_index < record.height and column_index < record.width else 'pad',
            'expected_class': expected_class,
            'predicted_class': predicted_class,
            'cell_mean_rgb': [round(float(value), 4) for value in means[row_index, column_index].tolist()],
            'expected_squared_distance': round(float(distances[expected_class].item()), 4),
            'predicted_squared_distance': round(float(distances[predicted_class].item()), 4),
            'nearest_second_margin': round(float((sorted_distances[1] - sorted_distances[0]).item()), 4),
            'second_nearest_class': int(sorted_indices[1].item()),
        })

    pad_precision_denominator = pad_true_positive + pad_false_positive
    pad_recall_denominator = pad_true_positive + pad_false_negative
    return {
        'record_id': record.record_id,
        'task_id': record.task_id,
        'split': record.split,
        'example_index': record.example_index,
        'height': record.height,
        'width': record.width,
        'area': record.height * record.width,
        'canvas_size': canvas_size,
        'prediction_valid': bool(prediction.valid),
        'invalid_reason': prediction.invalid_reason,
        'predicted_shape': predicted_shape,
        'shape_exact': shape_exact,
        'exact_grid': exact_grid,
        'grid_cells_correct': int(true_grid_matches.sum().item()),
        'grid_cells_total': int(true_grid_matches.numel()),
        'grid_cell_accuracy': float(true_grid_matches.to(dtype=torch.float32).mean().item()),
        'full_canvas_cells_correct': int(matches.sum().item()),
        'full_canvas_cells_total': int(matches.numel()),
        'full_canvas_cell_accuracy': float(matches.to(dtype=torch.float32).mean().item()),
        'pad_true_positive': pad_true_positive,
        'pad_false_positive': pad_false_positive,
        'pad_false_negative': pad_false_negative,
        'pad_precision': (
            pad_true_positive / pad_precision_denominator
            if pad_precision_denominator else 1.0
        ),
        'pad_recall': (
            pad_true_positive / pad_recall_denominator
            if pad_recall_denominator else 1.0
        ),
        'mismatch_count': len(mismatches),
        'mismatches': mismatches,
        'expected_grid': [list(row) for row in record.grid] if not exact_grid else None,
        'predicted_grid': (
            [list(row) for row in prediction.grid]
            if not exact_grid and prediction.grid is not None else None
        ),
    }


def summarize(rows: list[dict[str, Any]]) -> dict[str, Any]:
    count = len(rows)
    grid_correct = sum(row['grid_cells_correct'] for row in rows)
    grid_total = sum(row['grid_cells_total'] for row in rows)
    canvas_correct = sum(row['full_canvas_cells_correct'] for row in rows)
    canvas_total = sum(row['full_canvas_cells_total'] for row in rows)
    pad_tp = sum(row['pad_true_positive'] for row in rows)
    pad_fp = sum(row['pad_false_positive'] for row in rows)
    pad_fn = sum(row['pad_false_negative'] for row in rows)
    return {
        'count': count,
        'exact_grid_count': sum(int(row['exact_grid']) for row in rows),
        'exact_grid_accuracy': sum(int(row['exact_grid']) for row in rows) / count if count else None,
        'shape_exact_count': sum(int(row['shape_exact']) for row in rows),
        'shape_accuracy': sum(int(row['shape_exact']) for row in rows) / count if count else None,
        'grid_cell_accuracy': grid_correct / grid_total if grid_total else None,
        'full_canvas_cell_accuracy': canvas_correct / canvas_total if canvas_total else None,
        'pad_precision': pad_tp / (pad_tp + pad_fp) if pad_tp + pad_fp else 1.0,
        'pad_recall': pad_tp / (pad_tp + pad_fn) if pad_tp + pad_fn else 1.0,
        'failure_count': sum(int(not row['exact_grid']) for row in rows),
    }

## 5. Audit every output once on the 30×30 production canvas

Encoding streams 16 canvases at a time and decoding streams eight latents at a
time. No production output is repeated. Only a bounded number of failing RGB
reconstructions is retained in RAM for the later gallery.

In [ ]:
production_rows: list[dict[str, Any]] = []
gallery_cache: dict[str, dict[str, Any]] = {}
production_started = time.perf_counter()
torch.cuda.reset_peak_memory_stats(DEVICE)

for encode_start in tqdm(
    range(0, len(records), ENCODE_BATCH_SIZE),
    desc='Production 30x30 round trips',
    unit='encode batch',
):
    batch_records = records[encode_start:encode_start + ENCODE_BATCH_SIZE]
    target_images = torch.stack([render_grid_canvas(record.grid) for record in batch_records])
    clean_latents = vae_encoder(
        target_images,
        spatial_shape=PRODUCTION_SPATIAL_SHAPE,
    )
    del target_images

    for decode_start in range(0, len(batch_records), DECODE_BATCH_SIZE):
        decode_records = batch_records[decode_start:decode_start + DECODE_BATCH_SIZE]
        decoded = vae_decoder(
            clean_latents[decode_start:decode_start + len(decode_records)],
            spatial_shape=PRODUCTION_SPATIAL_SHAPE,
        )
        decoded_rgb = normalized_images_to_uint8(decoded)
        class_canvases = classify_arc_cells(decoded_rgb).to(device='cpu')
        predictions = decode_arc_grids(class_canvases)
        for local_index, record in enumerate(decode_records):
            row = analyze_reconstruction(
                record,
                decoded_rgb[local_index],
                class_canvases[local_index],
                predictions[local_index],
                canvas_size=CANVAS_SIZE,
            )
            production_rows.append(row)
            if not row['exact_grid'] and len(gallery_cache) < MAX_GALLERY_FAILURES:
                gallery_cache[record.record_id] = {
                    'record': record,
                    'production_rgb': decoded_rgb[local_index].clone(),
                    'production_classes': class_canvases[local_index].clone(),
                }
        del decoded, decoded_rgb, class_canvases, predictions
    del clean_latents

torch.cuda.synchronize(DEVICE)
production_seconds = time.perf_counter() - production_started
if len(production_rows) != EXPECTED_OUTPUT_COUNT:
    raise RuntimeError(
        f'Production result count mismatch: {len(production_rows)} != {EXPECTED_OUTPUT_COUNT}'
    )

production_summaries = {
    'all': summarize(production_rows),
    'train': summarize([row for row in production_rows if row['split'] == 'train']),
    'test': summarize([row for row in production_rows if row['split'] == 'test']),
}
production_failures = [row for row in production_rows if not row['exact_grid']]
print(json.dumps({
    'elapsed_seconds': round(production_seconds, 3),
    'peak_gpu_memory_GiB': round(torch.cuda.max_memory_allocated(DEVICE) / 2**30, 3),
    'summaries': production_summaries,
    'first_failure_ids': [row['record_id'] for row in production_failures[:20]],
}, indent=2))

## 6. Compact rerun of production failures only

Each failed output is rerun using `max(height, width)` as its minimum square
canvas. Rectangular grids retain only the unavoidable padding needed to form a
square. Recovery here supports padding sensitivity; failure in both paths points
to a palette/content-specific VAE limitation rather than batch execution.

In [ ]:
records_by_id = {record.record_id: record for record in records}
compact_jobs = []
failures_by_canvas_size: dict[int, list[OutputRecord]] = defaultdict(list)
for row in production_failures:
    record = records_by_id[row['record_id']]
    failures_by_canvas_size[max(record.height, record.width)].append(record)
for canvas_size, grouped_records in sorted(failures_by_canvas_size.items()):
    for start in range(0, len(grouped_records), ENCODE_BATCH_SIZE):
        compact_jobs.append((canvas_size, grouped_records[start:start + ENCODE_BATCH_SIZE]))

compact_by_record: dict[str, dict[str, Any]] = {}
compact_started = time.perf_counter()
for canvas_size, batch_records in tqdm(
    compact_jobs,
    desc='Compact failure round trips',
    unit='encode batch',
):
    spatial_shape = FluxSpatialShape(height=canvas_size, width=canvas_size)
    target_images = torch.stack([
        render_grid_canvas(record.grid, canvas_size=canvas_size)
        for record in batch_records
    ])
    clean_latents = vae_encoder(target_images, spatial_shape=spatial_shape)
    del target_images
    for decode_start in range(0, len(batch_records), DECODE_BATCH_SIZE):
        decode_records = batch_records[decode_start:decode_start + DECODE_BATCH_SIZE]
        decoded = vae_decoder(
            clean_latents[decode_start:decode_start + len(decode_records)],
            spatial_shape=spatial_shape,
        )
        decoded_rgb = normalized_images_to_uint8(decoded)
        class_canvases = classify_arc_cells(
            decoded_rgb,
            canvas_size=canvas_size,
            include_pad=True,
        ).to(device='cpu')
        predictions = decode_arc_grids(class_canvases, canvas_size=canvas_size)
        for local_index, record in enumerate(decode_records):
            compact_row = analyze_reconstruction(
                record,
                decoded_rgb[local_index],
                class_canvases[local_index],
                predictions[local_index],
                canvas_size=canvas_size,
            )
            compact_by_record[record.record_id] = compact_row
            if record.record_id in gallery_cache:
                gallery_cache[record.record_id].update({
                    'compact_rgb': decoded_rgb[local_index].clone(),
                    'compact_classes': class_canvases[local_index].clone(),
                    'compact_canvas_size': canvas_size,
                })
        del decoded, decoded_rgb, class_canvases, predictions
    del clean_latents

if compact_jobs:
    torch.cuda.synchronize(DEVICE)
compact_seconds = time.perf_counter() - compact_started
for row in production_rows:
    compact_row = compact_by_record.get(row['record_id'])
    row.update({
        'compact_canvas_size': compact_row['canvas_size'] if compact_row else None,
        'compact_exact_grid': compact_row['exact_grid'] if compact_row else None,
        'compact_shape_exact': compact_row['shape_exact'] if compact_row else None,
        'compact_mismatch_count': compact_row['mismatch_count'] if compact_row else None,
    })

compact_failures = list(compact_by_record.values())
print(json.dumps({
    'production_failures': len(production_failures),
    'compact_reruns': len(compact_failures),
    'compact_exact_grids': sum(int(row['exact_grid']) for row in compact_failures),
    'recovered_by_compact_canvas': sum(int(row['exact_grid']) for row in compact_failures),
    'still_failed_compact': [
        row['record_id'] for row in compact_failures if not row['exact_grid']
    ][:50],
    'elapsed_seconds': round(compact_seconds, 3),
}, indent=2))

## 7. Save tables, detailed failures, and bounded gallery pages

`per-output.csv` proves coverage of every original target. `failures.jsonl`
contains the complete cell-level differences without making the main report
unwieldy. Gallery pages are capped at 64 failures; tables are never truncated.

In [ ]:
from PIL import Image, ImageDraw

PER_OUTPUT_PATH = OUTPUT_DIR / 'per-output.csv'
FAILURES_CSV_PATH = OUTPUT_DIR / 'failures.csv'
FAILURES_JSONL_PATH = OUTPUT_DIR / 'failures.jsonl'
SUMMARY_CSV_PATH = OUTPUT_DIR / 'summary.csv'

flat_fields = [
    'record_id', 'task_id', 'split', 'example_index', 'height', 'width', 'area',
    'prediction_valid', 'invalid_reason', 'predicted_shape', 'shape_exact',
    'exact_grid', 'grid_cells_correct', 'grid_cells_total', 'grid_cell_accuracy',
    'full_canvas_cell_accuracy', 'pad_precision', 'pad_recall', 'mismatch_count',
    'compact_canvas_size', 'compact_exact_grid', 'compact_shape_exact',
    'compact_mismatch_count',
]


def csv_value(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return json.dumps(value, separators=(',', ':'))
    return value


def write_flat_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=flat_fields, extrasaction='ignore')
        writer.writeheader()
        for row in rows:
            writer.writerow({field: csv_value(row.get(field)) for field in flat_fields})


write_flat_csv(PER_OUTPUT_PATH, production_rows)
write_flat_csv(FAILURES_CSV_PATH, production_failures)
with FAILURES_JSONL_PATH.open('w', encoding='utf-8') as handle:
    for row in production_failures:
        detailed = dict(row)
        detailed['compact_diagnosis'] = compact_by_record.get(row['record_id'])
        handle.write(json.dumps(detailed, sort_keys=True) + '\n')

with SUMMARY_CSV_PATH.open('w', newline='', encoding='utf-8') as handle:
    summary_fields = ['group', *next(iter(production_summaries.values())).keys()]
    writer = csv.DictWriter(handle, fieldnames=summary_fields)
    writer.writeheader()
    for group, values in production_summaries.items():
        writer.writerow({'group': group, **values})


def class_rgb(class_canvas: torch.Tensor) -> np.ndarray:
    return TARGET_PALETTE_RGB[class_canvas].numpy().astype(np.uint8)


def expected_rgb(record: OutputRecord) -> np.ndarray:
    labels = torch.tensor(record.grid, dtype=torch.int64)
    return TARGET_PALETTE_RGB[labels].numpy().astype(np.uint8)


def fit_panel(array: np.ndarray, *, size: int = 160) -> Image.Image:
    image = Image.fromarray(array)
    image.thumbnail((size, size), Image.Resampling.NEAREST)
    panel = Image.new('RGB', (size, size), 'white')
    panel.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return panel


gallery_paths = []
gallery_entries = list(gallery_cache.values())
for page_index in range(math.ceil(len(gallery_entries) / GALLERY_ROWS_PER_PAGE)):
    page_entries = gallery_entries[
        page_index * GALLERY_ROWS_PER_PAGE:(page_index + 1) * GALLERY_ROWS_PER_PAGE
    ]
    row_height = 205
    page = Image.new('RGB', (710, row_height * len(page_entries)), 'white')
    draw = ImageDraw.Draw(page)
    for row_index, entry in enumerate(page_entries):
        record = entry['record']
        result = next(row for row in production_failures if row['record_id'] == record.record_id)
        y = row_index * row_height
        compact_result = compact_by_record[record.record_id]
        label = (
            f"{record.record_id} {record.height}x{record.width} | "
            f"prod mismatches={result['mismatch_count']} | "
            f"compact exact={compact_result['exact_grid']}"
        )
        draw.text((5, y + 3), label, fill='black')
        panel_titles = ('expected', 'production RGB', 'production classes', 'compact classes')
        crop_cells = min(CANVAS_SIZE, max(record.height, record.width) + 3)
        arrays = [
            expected_rgb(record),
            entry['production_rgb'][
                :, :crop_cells * CELL_SIZE, :crop_cells * CELL_SIZE
            ].permute(1, 2, 0).numpy(),
            class_rgb(entry['production_classes'][:crop_cells, :crop_cells]),
            class_rgb(entry['compact_classes']),
        ]
        for column_index, (title, array) in enumerate(zip(panel_titles, arrays, strict=True)):
            x = 5 + column_index * 175
            draw.text((x, y + 23), title, fill='black')
            page.paste(fit_panel(array), (x, y + 42))
    path = OUTPUT_DIR / f'failure-gallery-{page_index + 1:03d}.png'
    page.save(path)
    gallery_paths.append(path)

if gallery_paths:
    display(Image.open(gallery_paths[0]))
print(json.dumps({
    'per_output_csv': str(PER_OUTPUT_PATH),
    'failures_csv': str(FAILURES_CSV_PATH),
    'failures_jsonl': str(FAILURES_JSONL_PATH),
    'summary_csv': str(SUMMARY_CSV_PATH),
    'gallery_pages': [str(path) for path in gallery_paths],
}, indent=2))

## 8. Durable report and interpretation boundary

A 4,308/4,308 result establishes the clean original-output ceiling for this
dataset snapshot; it says nothing about augmented color/layout combinations or
noisy FLUX predictions. If failures recover on compact canvases, the evidence is
consistent with padding sensitivity. Failures that remain compact require
palette/content-level inspection.

In [ ]:
REPORT_PATH = OUTPUT_DIR / 'report.json'


def artifact_record(path: Path) -> dict[str, Any]:
    return {
        'path': str(path),
        'size_bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    }


if not production_failures:
    interpretation = (
        'All original labelled training outputs round-trip exactly through the '
        'production VAE path for this dataset, palette, and checkpoint snapshot.'
    )
elif compact_by_record and all(row['exact_grid'] for row in compact_by_record.values()):
    interpretation = (
        'Every production failure recovers on its minimum square canvas; this '
        'supports a padding-sensitive VAE limitation.'
    )
elif compact_by_record:
    interpretation = (
        'At least one production failure remains on its minimum square canvas; '
        'padding alone does not explain the complete failure set.'
    )
else:
    interpretation = 'No compact diagnosis was required because production had no failures.'

artifact_paths = [
    PER_OUTPUT_PATH,
    FAILURES_CSV_PATH,
    FAILURES_JSONL_PATH,
    SUMMARY_CSV_PATH,
    *gallery_paths,
]
report = {
    'schema_version': 1,
    'experiment': 'vae-training-output-roundtrip-audit',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source': {
        'repository_url': REPOSITORY_URL,
        'requested_commit': SOURCE_REVISION,
        'resolved_commit': RESOLVED_SOURCE_COMMIT,
    },
    'runtime': {
        'python': sys.version,
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'gpu': gpu.name,
        'gpu_memory_GiB': gpu.total_memory / 2**30,
        'peak_gpu_memory_GiB': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
    },
    'configuration': {
        'seed': SEED,
        'palette_id': TARGET_PALETTE_ID,
        'palette_rgb': TARGET_PALETTE_RGB.tolist(),
        'canvas_size': CANVAS_SIZE,
        'cell_size': CELL_SIZE,
        'image_size': CANVAS_SIZE * CELL_SIZE,
        'posterior_statistic': 'mode',
        'vae_compute_dtype': 'float32',
        'encode_batch_size': ENCODE_BATCH_SIZE,
        'decode_batch_size': DECODE_BATCH_SIZE,
        'include_inputs': False,
        'include_augmentations': False,
        'compact_rerun_scope': 'production failures only',
    },
    'artifacts': {
        'vae_source_path': str(VAE_SOURCE_PATH),
        'vae_sha256': vae_sha256,
        'training_data_dir': str(TRAINING_DATA_DIR),
        'dataset_sha256': DATASET_SHA256,
    },
    'dataset': dataset_summary,
    'task_manifest': task_manifest,
    'production': {
        'elapsed_seconds': production_seconds,
        'summaries': production_summaries,
        'failure_ids': [row['record_id'] for row in production_failures],
    },
    'compact_failure_diagnosis': {
        'elapsed_seconds': compact_seconds,
        'count': len(compact_by_record),
        'exact_grid_count': sum(int(row['exact_grid']) for row in compact_by_record.values()),
        'still_failed_ids': [
            row['record_id'] for row in compact_by_record.values() if not row['exact_grid']
        ],
    },
    'interpretation': interpretation,
    'artifacts_written': [artifact_record(path) for path in artifact_paths],
}
REPORT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print(json.dumps({
    'status': 'COMPLETE',
    'production_exact_grids': (
        f"{production_summaries['all']['exact_grid_count']}/"
        f"{production_summaries['all']['count']}"
    ),
    'production_failure_count': len(production_failures),
    'compact_recovered': sum(int(row['exact_grid']) for row in compact_by_record.values()),
    'compact_reruns': len(compact_by_record),
    'interpretation': interpretation,
    'report': str(REPORT_PATH),
}, indent=2))

## Next step

Use `report.json` as the aggregate source of truth and `failures.jsonl` for
cell-level investigation. A clean original-output result should be followed by
a separate augmentation audit before claiming that every generated training
target is VAE-safe.